# Gemini finetune test

In [88]:
import os
import json

In [89]:
def clean_data(data):
    # Your cleaning logic here
    # Clean sentences (Keep all entries within quotes ‘’), if not, keep the whole
    cleaned_data = []
    for entry in data:
        if entry == None:
            continue
        original = entry['source']
        translated = entry['translation']

        # Extract text within quotes if present
        if '‘' in original and '’' in original:
            start = original.index('‘') + 1
            end = original.index('’')
            original = original[start:end].strip()
        
        if '‘' in translated and '’' in translated:
            start = translated.index('‘') + 1
            end = translated.index('’')
            translated = translated[start:end].strip()

        # If "Translation: " prefix exists, remove it
        if translated.lower().startswith("translation:"):
            translated = translated[len("translation:"):].strip()
        
        if translated.lower().startswith("translation -"):
            translated = translated[len("translation -"):].strip()
        if translated.lower().startswith("translation in english:"):
            translated = translated[len("translation in english:"):].strip()

        # Do it for all other types of quotes as well
        quote_pairs = [('“', '”'), ('‘', '’'), ('"', '"'), ("'", "'")]
        for open_quote, close_quote in quote_pairs:
            if translated.startswith(open_quote) and translated.endswith(close_quote):
                translated = translated[1:-1].strip()
            if original.startswith(open_quote) and original.endswith(close_quote):
                original = original[1:-1].strip()
        cleaned_data.append({
            'source': original,
            'translation': translated
        })


    return cleaned_data

In [100]:
def get_gemini_finetune_json(train_data, test_data, base_filename, language_name):
    
    output_path = "../data/gemini_finetune_data"

    combined_output_path = os.path.join(output_path, language_name)
    os.makedirs(combined_output_path, exist_ok=True)
    output_filename = os.path.join(combined_output_path, base_filename + "_train.jsonl")

    with open(output_filename, "w", encoding="utf-8") as outfile:
        for entry in train_data:
            json_line = {
                "contents":[
                    {
                        "role": "user",
                        "parts": [
                            {"text": f"Translate to english from {language_name}: " + entry["source"]}
                        ]
                    },
                    {

                        "role": "model",
                        "parts": [
                            {"text": entry["translation"]}
                        ],
                    }
                ]
            }
            outfile.write(json.dumps(json_line) + "\n")

    print(f"Saved {len(train_data)} training entries to {output_filename}")


In [101]:
language_name = "sursilvan_romansh"
file_dir = f"../data/generated_results_batched/{language_name}"

for filename in os.listdir(file_dir):
    if filename.endswith(".json"):
        file_path = os.path.join(file_dir, filename)
        if file_path.endswith("test.jsonl"):
            continue
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        base_filename = filename.split(".json")[0]
        cleaned_data = clean_data(data)

        get_gemini_finetune_json(cleaned_data, cleaned_data, base_filename, language_name)

        print(f"File: {base_filename}, Number of entries: {len(data)}")

Saved 3879 training entries to ../data/gemini_finetune_data\sursilvan_romansh\sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_10_train.jsonl
File: sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_10, Number of entries: 3879
Saved 5814 training entries to ../data/gemini_finetune_data\sursilvan_romansh\sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_15_train.jsonl
File: sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_15, Number of entries: 5814
Saved 7754 training entries to ../data/gemini_finetune_data\sursilvan_romansh\sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_20_train.jsonl
File: sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_20, Number of entries: 7754
Saved 1937 training entries to ../data/gemini_finetune_data\sursilvan_romansh\sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_5_train.jsonl
File: sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_5, Number of entries: 1937
Saved 3879 training en

In [102]:
import json

In [103]:
# Create test set using original parallel sentences after 400

parallel_sents_file = f"../data/parallel_sentences/{language_name}_test_set_cleaned.json"

if os.path.exists(parallel_sents_file):
    with open(parallel_sents_file, "r", encoding="utf-8") as file:
        parallel_data = json.load(file)
    test_set = parallel_data
    print(f"Created test set with {len(test_set)} entries from parallel sentences.")
else:
    print(f"File {parallel_sents_file} does not exist.")

Created test set with 500 entries from parallel sentences.


In [104]:
test_set_sents_only = [
    {
        "source": entry["source"],
        "translation": entry["translation"]
    }
    for entry in test_set
]

In [105]:
test_set_sents_only[0], len(test_set_sents_only)

({'source': 'ˈɛləz ɔn traj ˈpɛra ˈʨɔmbəs […].',
  'translation': 'They [the ants] have three pairs of legs.'},
 500)

In [106]:
# save as test.jsonl
output_dir = os.path.join("../data/gemini_finetune_data", language_name)
output_filename = os.path.join(output_dir, "test.jsonl")
with open(output_filename, "w", encoding="utf-8") as outfile:  
    for entry in test_set_sents_only:
        json_line = {
            "contents":[
                {
                    "role": "user",
                    "parts": [
                        {"text": f"Translate to english from {language_name}: " + entry["source"]}
                    ]
                },
                {

                    "role": "model",
                    "parts": [
                        {"text": entry["translation"]}
                    ],
                }
            ]
        }
        outfile.write(json.dumps(json_line) + "\n")
print(f"Saved {len(test_set_sents_only)} testing entries to {output_filename}")





Saved 500 testing entries to ../data/gemini_finetune_data\sursilvan_romansh\test.jsonl


In [107]:
# Create test set for inference

inference_output_filename = os.path.join(output_dir, f"test_inference.jsonl")

with open(inference_output_filename, "w", encoding="utf-8") as outfile:
    for i, entry in enumerate(test_set_sents_only):
        json_line = {
            # "key" helps you re-sort the results later
            "key": f"{language_name}_sent_{i}", 
            "request": {
                "contents": [
                    {
                        "role": "user",
                        "parts": [
                            {"text": f"Translate to english from {language_name}: " + entry["source"]}
                        ]
                    }
                ]
            }
        }
        outfile.write(json.dumps(json_line) + "\n")

print(f"Saving {len(test_set_sents_only)} inference entries to {inference_output_filename}")

Saving 500 inference entries to ../data/gemini_finetune_data\sursilvan_romansh\test_inference.jsonl


# SDK Finetuning test

In [108]:
import time
from vertexai.tuning import sft
from google.cloud import aiplatform
from google.cloud import storage
from google.oauth2 import service_account
import vertexai
import os


In [109]:
project_id = "project-128068bc-0c84-4cfe-944"
location = "europe-west1"
bucket_name = "thesis_finetune_bucket_1"

In [110]:

data_folder = f"../data/gemini_finetune_data/{language_name}"
print(f"Data folder: {data_folder}")


train_files = []
test_file = None
for file in os.listdir(data_folder):
    print(file)
    if file.endswith("_train.jsonl"):
        train_files.append(os.path.join(data_folder, file))
    elif file == "test.jsonl":
        test_file = os.path.join(data_folder, file)
    elif file == "test_inference.jsonl":
        test_inference_file = os.path.join(data_folder, file)
print(f"Number of training files: {len(train_files)}")
print(f"Number of testing files: {1 if test_file else 0}")
print(f"Number of testing inference files: {1 if test_inference_file else 0}")

Data folder: ../data/gemini_finetune_data/sursilvan_romansh
sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_10_train.jsonl
sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_15_train.jsonl
sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_20_train.jsonl
sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_rule_batch_5_train.jsonl
sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_section_batch_10_train.jsonl
sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_section_batch_15_train.jsonl
sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_section_batch_20_train.jsonl
sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_section_batch_5_train.jsonl
sursilvan_romansh_all_batch_prompts_ADV_400_rule_batch_10_train.jsonl
sursilvan_romansh_all_batch_prompts_ADV_400_rule_batch_15_train.jsonl
sursilvan_romansh_all_batch_prompts_ADV_400_rule_batch_20_train.jsonl
sursilvan_romansh_all_batch_prompts_ADV_400_rule_batch_5_train.jsonl
sursilvan_romansh_all_batch_prompts_ADV_400_section_batch_1

In [111]:
with open(test_file, "r", encoding="utf-8") as f:
    test_file_data = f.readlines()
print(f"Number of entries in test file: {len(test_file_data)}")

Number of entries in test file: 500


In [112]:
import json
import pandas as pd
eval_list = []

for line in test_file_data:
    row = json.loads(line)
    prompt = row['contents'][0]['parts'][0]['text']
    response = row['contents'][1]['parts'][0]['text']
    evaluation_entry = {
        "prompt": prompt,
        "response": response
    }
    eval_list.append(evaluation_entry)

print(f"Number of evaluation entries: {len(eval_list)}")

Number of evaluation entries: 500


In [113]:
eval_df = pd.DataFrame(eval_list)
eval_df.head()

,prompt,response
0,Translate to english from sursilvan_romansh: ˈ...,They [the ants] have three pairs of legs.
1,Translate to english from sursilvan_romansh: A...,"Before bad weather comes, the cows run back an..."
2,Translate to english from sursilvan_romansh: Q...,He would fall asleep. And then I wasn’t able t...
3,Translate to english from sursilvan_romansh: P...,"As for mining, it could well be that the rock ..."
4,Translate to english from sursilvan_romansh: 1...,[that’s what it takes you to] go on foot.


In [114]:
service_account_key_file = "../api_keys/service_account_keys/service_account_key.json"

print(os.path.exists(service_account_key_file))

True


In [115]:
creds = service_account.Credentials.from_service_account_file(service_account_key_file)

In [116]:
vertexai.init(project=project_id, location=location,credentials=creds)
storage_client = storage.Client(credentials=creds,project=project_id)
bucket = storage_client.bucket(bucket_name)

In [117]:
def upload_to_gcs(local_path, gcs_path):
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    return f"gs://{bucket.name}/{gcs_path}"



In [118]:
# 1. Prepare common test set
test_set_uri = upload_to_gcs(test_file, f"{language_name}/tuning/test.jsonl")

print(f"Uploaded test set to {test_set_uri}")

Uploaded test set to gs://thesis_finetune_bucket_1/sursilvan_romansh/tuning/test.jsonl


In [119]:
test_set_inference_uri = upload_to_gcs(test_inference_file,f"{language_name}/tuning/test_inference.jsonl")

print(f"Uploaded test inference file to {test_set_inference_uri}")

Uploaded test inference file to gs://thesis_finetune_bucket_1/sursilvan_romansh/tuning/test_inference.jsonl


In [120]:
import os
import time
import pandas as pd
from vertexai.tuning import sft

In [121]:
project_id = "project-128068bc-0c84-4cfe-944"
location = "europe-west1"
bucket_name = "thesis_finetune_bucket_1"

In [122]:
service_account_key_file = "../api_keys/service_account_keys/service_account_key.json"

print(os.path.exists(service_account_key_file))

True


In [123]:
creds = service_account.Credentials.from_service_account_file(service_account_key_file)

In [124]:
vertexai.init(project=project_id, location=location,credentials=creds)
storage_client = storage.Client(credentials=creds,project=project_id)
bucket = storage_client.bucket(bucket_name)

In [125]:
import os
import time
import pandas as pd
from vertexai.tuning import sft
from google.cloud import aiplatform
from vertexai.evaluation import EvalTask



In [126]:
len(train_files)

32

In [127]:
df_tracking = pd.read_csv(TRACKING_FILE)
df_tracking["batch_name"].tolist()


['kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10',
 'kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15',
 'kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_5',
 'kalamang_all_batch_prompts_ADJ-NUM_400_section_batch_10',
 'kalamang_all_batch_prompts_ADJ-NUM_400_section_batch_15',
 'kalamang_all_batch_prompts_ADJ-NUM_400_section_batch_20',
 'kalamang_all_batch_prompts_ADJ-NUM_400_section_batch_5',
 'kalamang_all_batch_prompts_ADV_400_rule_batch_10',
 'kalamang_all_batch_prompts_ADV_400_rule_batch_15',
 'kalamang_all_batch_prompts_ADV_400_rule_batch_20',
 'kalamang_all_batch_prompts_ADV_400_rule_batch_5',
 'kalamang_all_batch_prompts_ADV_400_section_batch_10',
 'kalamang_all_batch_prompts_ADV_400_section_batch_15',
 'kalamang_all_batch_prompts_ADV_400_section_batch_20',
 'kalamang_all_batch_prompts_ADV_400_section_batch_5',
 'kalamang_all_batch_prompts_NOUN-PNOUN_400_rule_batch_10',
 'kalamang_all_batch_prompts_NOUN-PNOUN_400_rule_batch_15',
 'kalamang_all_batch_prompts_NOUN-PNO

In [45]:
if not df_tracking[df_tracking['batch_name'] == base_filename].empty:
    print("First IF")
    print(df_tracking.loc[df_tracking['batch_name'] == base_filename, 'status'].values[0])
    if df_tracking.loc[df_tracking['batch_name'] == base_filename, 'status'].values[0] == "TUNING_DONE_BATCH_SUBMITTED":
        print(f"Skipping {base_filename}, already completed.")

In [128]:

prev_batch_prediction_job = None

In [54]:
found_jobs = aiplatform.BatchPredictionJob.list(
                order_by="create_time desc"
            )

print(found_jobs)

resource name: projects/920145238348/locations/europe-west1/batchPredictionJobs/5816189805381287936, <google.cloud.aiplatform.jobs.BatchPredictionJob object at 0x0000017277B7E7B0> 
resource name: projects/920145238348/locations/europe-west1/batchPredictionJobs/7946392429127532544, <google.cloud.aiplatform.jobs.BatchPredictionJob object at 0x0000017277B7EA30> 
resource name: projects/920145238348/locations/europe-west1/batchPredictionJobs/5175552758387834880, <google.cloud.aiplatform.jobs.BatchPredictionJob object at 0x0000017277A84C20> 
resource name: projects/920145238348/locations/europe-west1/batchPredictionJobs/5149094110577033216, <google.cloud.aiplatform.jobs.BatchPredictionJob object at 0x0000017277A84E60> 
resource name: projects/920145238348/locations/europe-west1/batchPredictionJobs/894318362618757120, <google.cloud.aiplatform.jobs.BatchPredictionJob object at 0x0000017277A85010> 
resource name: projects/920145238348/locations/europe-west1/batchPredictionJobs/8149617362312626

BatchPredictionJob projects/920145238348/locations/europe-west1/batchPredictionJobs/5816189805381287936 current state:
3


In [ ]:


# --- CONFIGURATION ---
MAX_CONCURRENT_JOBS = 1  
TRACKING_FILE = f"{language_name}_tuning_experiments.csv"
TEST_SET_GCS_INFERENCE = test_inference_file

if os.path.exists(TRACKING_FILE):
    print(f"Tracking file {TRACKING_FILE} exists.")
else:
    print(f"Tracking file {TRACKING_FILE} does not exist. It will be created.")
    pd.DataFrame(columns=["batch_name", "tuning_job_id", "tuned_model_resource", "batch_job_id", "status", "timestamp"]).to_csv(TRACKING_FILE, index=False)

def get_active_jobs_count():
    jobs = sft.SupervisedTuningJob.list()
    active_states = ["JOB_STATE_RUNNING", "JOB_STATE_PENDING", "JOB_STATE_QUEUED"]
    return len([j for j in jobs if j.state.name in active_states])

skip_count = 0

for file in train_files[5:]:  # 5 file already done
    base_filename = os.path.basename(file).split("_train.jsonl")[0]
    print(f"Processing file: {base_filename}")
    
    # 1. Skip if already done
    df_tracking = pd.read_csv(TRACKING_FILE)
    if not df_tracking[df_tracking['batch_name'] == base_filename].empty:
        if df_tracking.loc[df_tracking['batch_name'] == base_filename, 'status'].values[0] == "TUNING_DONE_BATCH_SUBMITTED":
            print(f"Skipping {base_filename}, already completed.")
            continue

    print(f"Starting tuning for {base_filename}...")
    # 2. Wait for slot
    while get_active_jobs_count() >= MAX_CONCURRENT_JOBS:
        print("Waiting for slot...")
        time.sleep(120)

    # 3. Start Tuning (Sync=False to prevent widget error)
    gcs_train_path = upload_to_gcs(file, f"{language_name}/tuning/{base_filename}.jsonl")

    if skip_count > 0:
        print(f"Skipping tuning for {base_filename} as per skip count.")
        skip_count -= 1
    else:
        tuning_job = sft.train(
            source_model="gemini-2.5-flash",
            train_dataset=gcs_train_path,
            validation_dataset=test_set_uri,
            tuned_model_display_name=f"tuned-{base_filename}",
            # Hyperparameters
            epochs=3,
            adapter_size=1,

        )

    
        # 4. Wait for this specific job to finish so we can run inference
        print(f"Tuning {base_filename}... Job ID: {tuning_job.resource_name}")
        # Polling loop as per documentation
        while not tuning_job.has_ended:
            time.sleep(120)
            tuning_job.refresh()
            print(f"Job running... State: {tuning_job.state.name}")

        # Once finished, get the model resource
        
        tuned_model_resource = tuning_job.tuned_model_name 
        tuned_model_endpoint = tuning_job.tuned_model_endpoint_name
        model = aiplatform.Model(tuned_model_resource)

   # Check if the tuning job actually succeeded before proceeding
    if tuning_job.state.name != "JOB_STATE_SUCCEEDED":
        print(f"Tuning job failed or cancelled. State: {tuning_job.state.name}")
        # Handle failure (e.g., raise error or skip)
    else:
        print("Tuning job succeeded. Preparing for batch prediction...")
        
        # Get the model resource name (e.g., projects/123/locations/us-central1/models/456)
        tuned_model_resource = tuning_job.tuned_model_name 
        
        # Note: You do NOT need an endpoint for Batch Prediction, just the Model resource.
        # This saves you from having to deploy it to an endpoint ($$$).
        model = aiplatform.Model(tuned_model_resource)

        print(" Previous Job completed successfully. Waiting 3 minutes to ensure model is ready...")
        time.sleep(180)  # Small delay to ensure model is ready
        # 3. Run Batch Prediction
        # We use sync=False so the script submits the job and moves on, 
        # rather than blocking here for hours.

        print(f"Submitting batch prediction job for {base_filename}...")
        
        try:
            batch_prediction_job = model.batch_predict(
                job_display_name=f"inference-eval-{base_filename}",
                gcs_source=test_set_inference_uri,
                gcs_destination_prefix=f"gs://{bucket_name}/{language_name}/inference_results/{base_filename}/",
                instances_format="jsonl",
                predictions_format="jsonl",
                sync=False
            )
        except Exception as e:
            print(f"⚠️ ({e})")
            time.sleep(30) # Give Google a moment to register the job



        # ... (The rest of your code that updates the CSV works fine now) ...
        
        # Wait for the resource name to become available locally
        max_retries = 5
        for i in range(max_retries):
            try:
                # Try to access the name
                job_id = batch_prediction_job.resource_name
                print(f"Batch job submitted successfully: {job_id}")
                break # It worked! Exit the loop
            except RuntimeError:
                if i == max_retries - 1:
                    raise # If it fails 5 times, let it crash
                print("Waiting for job ID to populate...")
                time.sleep(60) # Wait 60 seconds and try again


        # 4. Update tracking file
        new_entry = {
            "batch_name": base_filename,
            "tuning_job_id": tuning_job.resource_name,
            "tuned_model_resource": tuned_model_resource,
            "batch_job_id": batch_prediction_job.resource_name, # Log the batch job ID too
            "status": "TUNING_DONE_BATCH_SUBMITTED",
            "timestamp": pd.Timestamp.now().isoformat()
        }
        
        # Safe append to CSV
        try:
            df_current = pd.read_csv(TRACKING_FILE)
        except FileNotFoundError:
            df_current = pd.DataFrame()
            
        df_tracking = pd.concat([df_current, pd.DataFrame([new_entry])], ignore_index=True)
        df_tracking.to_csv(TRACKING_FILE, index=False)
        print("Tracking file updated.")

Tracking file sursilvan_romansh_tuning_experiments.csv does not exist. It will be created.
Processing file: sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_section_batch_15
Starting tuning for sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_section_batch_15...
Creating SupervisedTuningJob
SupervisedTuningJob created. Resource name: projects/920145238348/locations/europe-west1/tuningJobs/2801979832903139328
To use this SupervisedTuningJob in another session:
tuning_job = sft.SupervisedTuningJob('projects/920145238348/locations/europe-west1/tuningJobs/2801979832903139328')
View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/europe-west1/tuning/tuningJob/2801979832903139328?project=920145238348


Tuning sursilvan_romansh_all_batch_prompts_ADJ-NUM_400_section_batch_15... Job ID: projects/920145238348/locations/europe-west1/tuningJobs/2801979832903139328


Job running... State: JOB_STATE_RUNNING


In [67]:
import os
from google.cloud import storage
from concurrent.futures import ThreadPoolExecutor

def download_blob(blob, bucket_name, prefix, local_destination):
    """Downloads a single blob, preserving folder structure."""
    
    # Construct the local path
    # We strip the prefix to get the relative path inside the folder
    relative_path = blob.name[len(prefix):] 
    
    # Remove leading slashes if any to avoid absolute path issues
    if relative_path.startswith("/"):
        relative_path = relative_path[1:]

    relative_path = relative_path.split("/")[0]
        
    relative_path = relative_path.replace(":", "-")
    print(f"Relative path: {relative_path}")
    # If the blob is just a folder placeholder, skip it
    if not relative_path:
        return

    local_path = os.path.join(local_destination, relative_path)

    # Create local directories if they don't exist
    local_dir = os.path.dirname(local_path)
    if not os.path.exists(local_dir):
        os.makedirs(local_dir, exist_ok=True)

    # Download
    print(f"Downloading {blob.name} to {local_path}...")
    blob.download_to_filename(local_path)

def download_folder_parallel(bucket_name, source_folder, local_destination=".", workers=8):
    """
    Downloads a folder from GCS recursively and in parallel.
    workers=8 mimics the default behavior of gsutil -m.
    """  

    # Ensure source folder ends with / so we don't download partial matches
    if not source_folder.endswith("/"):
        source_folder += "/"

    # 1. List all blobs
    print("Listing blobs...")
    blobs = list(bucket.list_blobs(prefix=source_folder))
    print(f"Found {len(blobs)} files.")

    # 2. Download in parallel
    with ThreadPoolExecutor(max_workers=workers) as executor:
        # We submit tasks to the pool
        futures = [
            executor.submit(download_blob, blob, bucket_name, source_folder, local_destination)
            for blob in blobs
        ]
        
        # Optional: Wait for all to finish and catch errors
        for future in futures:
            try:
                future.result()
            except Exception as e:
                print(f"❌ Error downloading file: {e}")



In [68]:
# --- USAGE ---
# Corresponds to: gsutil -m cp -r gs://my-bucket/data/images .
download_folder_parallel(
    bucket_name="thesis_finetune_bucket_1", 
    source_folder=f"{language_name}/inference_results", 
    local_destination=f"../data/gemini_finetune_inference_results/{language_name}"
)

Listing blobs...
Found 99 files.
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_5
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_5
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_5
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_section_batch_10
Relative path: kalamang_all_batch_prompts_ADJ-NUM_400_section_batch_10
Relative pat

In [74]:
count = 0

from shutil import copyfile

target_folder = f"../data/gemini_final_test_generations/{language_name}"

for folder, subfolders, files in os.walk(f"../data/gemini_finetune_inference_results/{language_name}"):
    for file in files:
        if file.endswith(".jsonl"):
            file_path = os.path.join(folder, file)
            print(f"Processing file: {file_path}")
            base_folder = os.path.basename(folder)
            base_folder = base_folder.split(".")[0]
            print(f"Base folder: {base_folder}")
            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
            target_file_path = os.path.join(target_folder, f"{base_folder}.jsonl")
            os.makedirs(target_folder, exist_ok=True)
            copyfile(file_path, target_file_path)

            #print(f"Number of lines in {file}: {len(lines)}")
            count += 1
print(f"Total files processed: {count}")

Processing file: ../data/gemini_finetune_inference_results/kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10\prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52.007487Z\predictions.jsonl
Base folder: prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10-2025-12-24T22-00-52
Processing file: ../data/gemini_finetune_inference_results/kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15\prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15-2025-12-24T22-48-29.359187Z\predictions.jsonl
Base folder: prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15-2025-12-24T22-48-29
Processing file: ../data/gemini_finetune_inference_results/kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20\prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20-2025-12-24T23-35-27.330502Z\predictions.jsonl
Base folder: prediction-tuned-kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20-